# Feature Extraction / Creation for modeling

In [95]:
import pandas as pd
import numpy as np

data = pd.read_csv('/home/spandan/Documents/sumago/Traffic-Prediction/dataset/dataset.csv')
data.head()

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,NaN,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,NaN,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,NaN,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,NaN,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,NaN,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


In [96]:
data['date_time'] = pd.to_datetime(data['date_time'])
data['date_time'].info

<bound method Series.info of 0       2012-10-02 09:00:00
1       2012-10-02 10:00:00
2       2012-10-02 11:00:00
3       2012-10-02 12:00:00
4       2012-10-02 13:00:00
                ...        
48199   2018-09-30 19:00:00
48200   2018-09-30 20:00:00
48201   2018-09-30 21:00:00
48202   2018-09-30 22:00:00
48203   2018-09-30 23:00:00
Name: date_time, Length: 48204, dtype: datetime64[us]>

ill remove holdays column as they are allmost empty so instead of making the data biassed or keeping the useless column , ill consider removing it.  
the weather description has a lot of text , and its useless too , so lets just straight way remove it.

In [97]:
data.drop(columns=['holiday' , 'weather_description'] , inplace=True)
data.columns

Index(['temp', 'rain_1h', 'snow_1h', 'clouds_all', 'weather_main', 'date_time',
       'traffic_volume'],
      dtype='str')

in EDA , we found out 2 rows with 0 as a traffic volume , it dosent really matter but they are still outliers , we will remove them

In [98]:
data = data[data['traffic_volume'] > 0]

let me quickly make all the columns we made for ourselfs in EDA :
1. hours
2. days_of_week
3. month
4. day
5. year

In [99]:
data['hour'] = data['date_time'].dt.hour
data['days_of_week'] = data['date_time'].dt.dayofweek
data['month'] = data['date_time'].dt.month
data['day'] = data['date_time'].dt.day
data['year'] = data['date_time'].dt.year

In [100]:
data.head()

,temp,rain_1h,snow_1h,clouds_all,weather_main,date_time,traffic_volume,hour,days_of_week,month,day,year
0,288.28,0.0,0.0,40,Clouds,2012-10-02 09:00:00,5545,9,1,10,2,2012
1,289.36,0.0,0.0,75,Clouds,2012-10-02 10:00:00,4516,10,1,10,2,2012
2,289.58,0.0,0.0,90,Clouds,2012-10-02 11:00:00,4767,11,1,10,2,2012
3,290.13,0.0,0.0,90,Clouds,2012-10-02 12:00:00,5026,12,1,10,2,2012
4,291.14,0.0,0.0,75,Clouds,2012-10-02 13:00:00,4918,13,1,10,2,2012


we can also keep weekends 

In [101]:
data['is_weekend'] = (data['day'].isin([5,6])).astype(int)
data['is_weekend'].value_counts()

is_weekend
0    44961
1     3241
Name: count, dtype: int64

In [102]:
data.head()

,temp,rain_1h,snow_1h,clouds_all,weather_main,date_time,traffic_volume,hour,days_of_week,month,day,year,is_weekend
0,288.28,0.0,0.0,40,Clouds,2012-10-02 09:00:00,5545,9,1,10,2,2012,0
1,289.36,0.0,0.0,75,Clouds,2012-10-02 10:00:00,4516,10,1,10,2,2012,0
2,289.58,0.0,0.0,90,Clouds,2012-10-02 11:00:00,4767,11,1,10,2,2012,0
3,290.13,0.0,0.0,90,Clouds,2012-10-02 12:00:00,5026,12,1,10,2,2012,0
4,291.14,0.0,0.0,75,Clouds,2012-10-02 13:00:00,4918,13,1,10,2,2012,0


## previous hour classification

so what i noticed is , traffic at current hour is highly influenced by the hour before current , or simply previous hour.  
let me just create a column which will store traffic for the previous hour of current hour's traffic. this will help in m2 classification

In [103]:
data = data.sort_values(by = 'date_time' , ascending=True)

In [104]:
data.head()

,temp,rain_1h,snow_1h,clouds_all,weather_main,date_time,traffic_volume,hour,days_of_week,month,day,year,is_weekend
0,288.28,0.0,0.0,40,Clouds,2012-10-02 09:00:00,5545,9,1,10,2,2012,0
1,289.36,0.0,0.0,75,Clouds,2012-10-02 10:00:00,4516,10,1,10,2,2012,0
2,289.58,0.0,0.0,90,Clouds,2012-10-02 11:00:00,4767,11,1,10,2,2012,0
3,290.13,0.0,0.0,90,Clouds,2012-10-02 12:00:00,5026,12,1,10,2,2012,0
4,291.14,0.0,0.0,75,Clouds,2012-10-02 13:00:00,4918,13,1,10,2,2012,0


In [105]:
data['volume_lag_1h'] = data['traffic_volume'].shift(1) #this basically shifts the data 1 row previous

In [106]:
data.head()

,temp,rain_1h,snow_1h,clouds_all,weather_main,date_time,traffic_volume,hour,days_of_week,month,day,year,is_weekend,volume_lag_1h
0,288.28,0.0,0.0,40,Clouds,2012-10-02 09:00:00,5545,9,1,10,2,2012,0,NaN
1,289.36,0.0,0.0,75,Clouds,2012-10-02 10:00:00,4516,10,1,10,2,2012,0,5545.0
2,289.58,0.0,0.0,90,Clouds,2012-10-02 11:00:00,4767,11,1,10,2,2012,0,4516.0
3,290.13,0.0,0.0,90,Clouds,2012-10-02 12:00:00,5026,12,1,10,2,2012,0,4767.0
4,291.14,0.0,0.0,75,Clouds,2012-10-02 13:00:00,4918,13,1,10,2,2012,0,5026.0


so what we did just now ? 
1. we created a column called lag by 1hour
2. stored the value of previous hour in it.
3. we sorted the data before doing it in order to get the previous data !

In [107]:
data[['date_time' , 'traffic_volume' , 'volume_lag_1h']].tail()

,date_time,traffic_volume,volume_lag_1h
48199,2018-09-30 19:00:00,3543,3947.0
48200,2018-09-30 20:00:00,2781,3543.0
48201,2018-09-30 21:00:00,2159,2781.0
48202,2018-09-30 22:00:00,1450,2159.0
48203,2018-09-30 23:00:00,954,1450.0


similarly , ill create lag by 6h , and 24h

In [108]:
data['volume_lag_6h'] = data['traffic_volume'].shift(6)
data['volume_lag_24h'] = data['traffic_volume'].shift(24)

In [109]:
data[
    [
        "date_time",
        "traffic_volume",
        "volume_lag_1h",
        "volume_lag_6h",
        "volume_lag_24h"
    ]
].head(30)

,date_time,traffic_volume,volume_lag_1h,volume_lag_6h,volume_lag_24h
0,2012-10-02 09:00:00,5545,NaN,NaN,NaN
1,2012-10-02 10:00:00,4516,5545.0,NaN,NaN
2,2012-10-02 11:00:00,4767,4516.0,NaN,NaN
3,2012-10-02 12:00:00,5026,4767.0,NaN,NaN
4,2012-10-02 13:00:00,4918,5026.0,NaN,NaN
5,2012-10-02 14:00:00,5181,4918.0,NaN,NaN
6,2012-10-02 15:00:00,5584,5181.0,5545.0,NaN
7,2012-10-02 16:00:00,6015,5584.0,4516.0,NaN
8,2012-10-02 17:00:00,5791,6015.0,4767.0,NaN
9,2012-10-02 18:00:00,4770,5791.0,5026.0,NaN


now that we have times in shifts of 24h , lets just create average time columns in those hours.  
we also need a standard deviation column for same !

In [110]:
data['roll_mean_24h'] = data['traffic_volume'].rolling(window = 24).mean()
data['roll_std_24h'] = data['traffic_volume'].rolling(window = 24).std()

In [111]:
data.head(30)

,temp,rain_1h,snow_1h,clouds_all,weather_main,date_time,traffic_volume,hour,days_of_week,month,day,year,is_weekend,volume_lag_1h,volume_lag_6h,volume_lag_24h,roll_mean_24h,roll_std_24h
0,288.28,0.0,0.0,40,Clouds,2012-10-02 09:00:00,5545,9,1,10,2,2012,0,NaN,NaN,NaN,NaN,NaN
1,289.36,0.0,0.0,75,Clouds,2012-10-02 10:00:00,4516,10,1,10,2,2012,0,5545.0,NaN,NaN,NaN,NaN
2,289.58,0.0,0.0,90,Clouds,2012-10-02 11:00:00,4767,11,1,10,2,2012,0,4516.0,NaN,NaN,NaN,NaN
3,290.13,0.0,0.0,90,Clouds,2012-10-02 12:00:00,5026,12,1,10,2,2012,0,4767.0,NaN,NaN,NaN,NaN
4,291.14,0.0,0.0,75,Clouds,2012-10-02 13:00:00,4918,13,1,10,2,2012,0,5026.0,NaN,NaN,NaN,NaN
5,291.72,0.0,0.0,1,Clear,2012-10-02 14:00:00,5181,14,1,10,2,2012,0,4918.0,NaN,NaN,NaN,NaN
6,293.17,0.0,0.0,1,Clear,2012-10-02 15:00:00,5584,15,1,10,2,2012,0,5181.0,5545.0,NaN,NaN,NaN
7,293.86,0.0,0.0,1,Clear,2012-10-02 16:00:00,6015,16,1,10,2,2012,0,5584.0,4516.0,NaN,NaN,NaN
8,294.14,0.0,0.0,20,Clouds,2012-10-02 17:00:00,5791,17,1,10,2,2012,0,6015.0,4767.0,NaN,NaN,NaN
9,293.10,0.0,0.0,20,Clouds,2012-10-02 18:00:00,4770,18,1,10,2,2012,0,5791.0,5026.0,NaN,NaN,NaN


In [112]:
data.isnull().sum()

temp               0
rain_1h            0
snow_1h            0
clouds_all         0
weather_main       0
date_time          0
traffic_volume     0
hour               0
days_of_week       0
month              0
day                0
year               0
is_weekend         0
volume_lag_1h      1
volume_lag_6h      6
volume_lag_24h    24
roll_mean_24h     23
roll_std_24h      23
dtype: int64

well this is quite huge dataset , so if we remove the nans caused due to us performing operations , we will just remove them

In [113]:
data = data.dropna()
data.shape

(48178, 18)

In [114]:
data.isnull().sum()

temp              0
rain_1h           0
snow_1h           0
clouds_all        0
weather_main      0
date_time         0
traffic_volume    0
hour              0
days_of_week      0
month             0
day               0
year              0
is_weekend        0
volume_lag_1h     0
volume_lag_6h     0
volume_lag_24h    0
roll_mean_24h     0
roll_std_24h      0
dtype: int64

now , in reality (cyclic time), 23:00 and 00:00 are 1 hour apart,  
but as per data , numerically , they are 23 hours apart. so we need to bring them closer . to do that we will  
  
1. place every hour on a circle. 
2. convert the circle positions of each one into numbers by sin and cos angles.
  
a circle is a 2 pie radiens or 360 degrees. soo we can compute the position using formula something like
2* pie * hour/24 , where hour/24 is the where the current hour lies in a day  
that equation will convert the hour into an angle ! 

so the angles and hours wil work something like  
hour = 0 , angle = 0  
hour = 6 , angle = 90 degree  
hour = 12 , angle = 180 degree  
hour = 18 , angle = 270 degree,  
and then to make then numerical , we will take there sin !

but sin cant handle the whole thing itself , as sin 0 = 0 and sin 180 also 0
soo we can add cos , they both can carry the whole thing out

In [115]:
data['hour_sin'] = np.sin(2* np.pi * data['hour']/24)
data['hour_cos'] = np.cos(2*  np.pi * data['hour']/24)

In [116]:
data.dtypes

temp                     float64
rain_1h                  float64
snow_1h                  float64
clouds_all                 int64
weather_main                 str
date_time         datetime64[us]
traffic_volume             int64
hour                       int32
days_of_week               int32
month                      int32
day                        int32
year                       int32
is_weekend                 int64
volume_lag_1h            float64
volume_lag_6h            float64
volume_lag_24h           float64
roll_mean_24h            float64
roll_std_24h             float64
hour_sin                 float64
hour_cos                 float64
dtype: object

we already have extracted what we needed from the date time , i think its good to go , we will just remove it at this point.  
what we gotto do now is encode the weather main.

In [117]:
data = data.drop(columns=['date_time'])
data.columns

Index(['temp', 'rain_1h', 'snow_1h', 'clouds_all', 'weather_main',
       'traffic_volume', 'hour', 'days_of_week', 'month', 'day', 'year',
       'is_weekend', 'volume_lag_1h', 'volume_lag_6h', 'volume_lag_24h',
       'roll_mean_24h', 'roll_std_24h', 'hour_sin', 'hour_cos'],
      dtype='str')

In [118]:
data['weather_main'].unique()

<StringArray>
[       'Clear',       'Clouds',         'Rain',      'Drizzle',
         'Mist',         'Haze',          'Fog', 'Thunderstorm',
         'Snow',       'Squall',        'Smoke']
Length: 11, dtype: str

so we have 11 categories in weather main , we can encode it ! lets go for one hot encoding  
we will drop the redundants so it can go well with linear models too

smoke and sqall dont have that many rows in them so lets just combine them and form a 'others' column

In [119]:
rare_weather = ['smoke' , 'squall']
data['weather_main'] = data['weather_main'].replace(
    rare_weather,
    'other'
)

In [120]:
data = pd.get_dummies(
    data,
    columns=['weather_main'],
    drop_first=True
)

In [121]:
data.info()

<class 'pandas.DataFrame'>
Index: 48178 entries, 24 to 48203
Data columns (total 28 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   temp                       48178 non-null  float64
 1   rain_1h                    48178 non-null  float64
 2   snow_1h                    48178 non-null  float64
 3   clouds_all                 48178 non-null  int64  
 4   traffic_volume             48178 non-null  int64  
 5   hour                       48178 non-null  int32  
 6   days_of_week               48178 non-null  int32  
 7   month                      48178 non-null  int32  
 8   day                        48178 non-null  int32  
 9   year                       48178 non-null  int32  
 10  is_weekend                 48178 non-null  int64  
 11  volume_lag_1h              48178 non-null  float64
 12  volume_lag_6h              48178 non-null  float64
 13  volume_lag_24h             48178 non-null  float64
 14  roll_

In [122]:
corr = data.corr(numeric_only=True)
corr['traffic_volume'].sort_values(ascending=False)

traffic_volume               1.000000
volume_lag_1h                0.912698
volume_lag_24h               0.369445
hour                         0.352566
roll_std_24h                 0.231593
roll_mean_24h                0.162638
temp                         0.130316
weather_main_Clouds          0.122080
volume_lag_6h                0.094021
clouds_all                   0.067044
weather_main_Haze            0.021001
weather_main_Rain            0.010792
year                         0.004926
rain_1h                      0.004715
weather_main_Drizzle         0.003086
snow_1h                      0.000734
is_weekend                   0.000290
weather_main_Smoke          -0.000227
month                       -0.002607
weather_main_Squall         -0.005495
day                         -0.007403
weather_main_Thunderstorm   -0.019245
weather_main_Snow           -0.030812
weather_main_Fog            -0.038880
weather_main_Mist           -0.061753
days_of_week                -0.149443
hour_sin    

the features we made are actually gold , they are greatly correlated.

In [123]:
data.to_csv('../dataset/feature_engineered_data.csv' , index=False)